# Awesome Visualization V2：規格表當主體

兩本課問的不是同一件事。

| | 這本 | `car_market_eda.ipynb` |
|---|---|---|
| 主表 | `Automobile_data.csv`（205 列、26 欄） | `car_data/` 九個品牌檔（清完約 9.7 萬列） |
| 問題 | 規格／保險欄怎麼拉開**目錄標價** | 英國上架價被什麼拉開 |
| 這份表有、那份沒有 | `symboling`、`horsepower`、`bore`／`stroke`、車身／驅動 | `model`、`year`、`mileage` |
| 圖 | seaborn + plotly（205 點可以全畫） | matplotlib + seaborn（大表要抽樣） |

舊 Altair 本在 `legacy/`。那本先走規格表、再看 Audi／Ford 上架檔，方向對，但兩表當成同一條故事、也沒講不能 concat。

這本把規格表講完，最後才把英國上架檔**聚合成品牌列**接進來。不 concat、不對到同一台車。

## 0. 載入規格表

[UCI / Kaggle automobile-dataset](https://www.kaggle.com/datasets/toramky/automobile-dataset)。`price` 是目錄標價（字串，有 `?`），單位跟英國上架 GBP 不是同一件事。

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    import plotly.express as px
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("plotly 沒裝，互動圖改用 seaborn。pip install plotly")

try:
    import missingno as msno
    HAS_MSNO = True
except ImportError:
    HAS_MSNO = False

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 110,
    "axes.titleweight": "bold",
    "font.sans-serif": ["Microsoft JhengHei", "DejaVu Sans"],
    "axes.unicode_minus": False,
})

auto_raw = pd.read_csv("Automobile_data.csv")
print(auto_raw.shape)
print(auto_raw.columns.tolist())
print("makes", auto_raw["make"].nunique(), sorted(auto_raw["make"].unique()))
print("no model column:", "model" not in auto_raw.columns)
auto_raw.head(3)

## 1. 缺值長在字元 `?` 上（M2／M3）

pandas 預設不會把 `?` 當 NaN。`normalized-losses` 缺最多（保險損失，約 1/5），整表丟掉會先砍掉一截。價格缺 4 列：後面凡是「對價格作圖」用 `priced`，規格本身仍留著。

In [ ]:
auto = auto_raw.replace("?", np.nan).copy()
print((auto.isna().sum()[auto.isna().sum() > 0]).sort_values(ascending=False).to_string())

if HAS_MSNO:
    msno.bar(auto, figsize=(10, 4))
    plt.title("Missing after treating '?' as NA")
    plt.tight_layout()
    plt.show()
else:
    auto.isna().mean().sort_values(ascending=False).head(8).plot(kind="bar")
    plt.ylabel("missing fraction")
    plt.title("Missing after treating '?' as NA")
    plt.tight_layout()
    plt.show()

## 2. 型別與分析用子集

數字欄先 `to_numeric`。`priced` = 有標價的列，後面畫價格只用它。

In [ ]:
NUM_COLS = [
    "symboling", "normalized-losses", "wheel-base", "length", "width", "height",
    "curb-weight", "engine-size", "bore", "stroke", "compression-ratio",
    "horsepower", "peak-rpm", "city-mpg", "highway-mpg", "price",
]
for c in NUM_COLS:
    auto[c] = pd.to_numeric(auto[c], errors="coerce")

priced = auto.dropna(subset=["price"]).copy()
print(f"priced rows {len(priced)} / {len(auto)} (dropped {auto['price'].isna().sum()} without sticker)")
print(priced[NUM_COLS].describe().T[["count", "mean", "50%", "min", "max"]].round(2))

## 3. 特徵工程（對齊模組，複合看目錄價）

| 欄 | 做法 | 模組 | 為什麼 |
|---|---|---|---|
| `num_doors`／`num_cylinders` | two／four → 2／4 | M4／M6 | 原文是字，後面才能算排量 |
| `disp_from_geom` | π × (bore/2)² × stroke × 缸數 | M6 | `bore`／`stroke` 單位是吋，結果接近 `engine-size`（c.i.） |
| `log_price` | `log1p(price)` | M5 | 標價右偏 |
| `hp_per_lb` | horsepower／curb-weight | M6 | 單位重馬力 |
| `make_freq` | 該品牌列數／N | M4 | 頻率編碼，**沒看價格** |
| `hp_bin`／`engine_bin` | `pd.cut` | M4 | 舊 V2 用來上色的那層 |
| `price_iqr_flag` | 價格 IQR 外 | M3 | 離群是超跑還是錯值 |

不做 Target Encoding：205 列、22 個 `make`，沒切摺就洩漏。

In [ ]:
WORD_NUM = {
    "two": 2, "three": 3, "four": 4, "five": 5,
    "six": 6, "eight": 8, "twelve": 12,
}

spec = priced.copy()
spec["num_doors"] = spec["num-of-doors"].map(WORD_NUM)
spec["num_cylinders"] = spec["num-of-cylinders"].map(WORD_NUM)
spec["disp_from_geom"] = (
    np.pi * (spec["bore"] / 2) ** 2 * spec["stroke"] * spec["num_cylinders"]
)
spec["log_price"] = np.log1p(spec["price"])
spec["hp_per_lb"] = spec["horsepower"] / spec["curb-weight"]
spec["make_freq"] = spec["make"].map(spec["make"].value_counts() / len(spec))
spec["hp_bin"] = pd.cut(
    spec["horsepower"], bins=[0, 80, 110, 160, 400],
    labels=["low", "mid", "high", "very_high"],
)
spec["engine_bin"] = pd.cut(
    spec["engine-size"], bins=[0, 100, 140, 200, 400],
    labels=["small", "medium", "large", "xl"],
)

q1, q3 = spec["price"].quantile([0.25, 0.75])
iqr = q3 - q1
spec["price_iqr_flag"] = (spec["price"] < q1 - 1.5 * iqr) | (spec["price"] > q3 + 1.5 * iqr)

check = spec.dropna(subset=["disp_from_geom", "engine-size"])
print(
    "geom vs engine-size corr",
    round(check["disp_from_geom"].corr(check["engine-size"]), 3),
    "| rows used", len(check),
)
print("IQR price outliers", int(spec["price_iqr_flag"].sum()))
print(spec[["price", "log_price", "engine-size", "disp_from_geom", "horsepower", "hp_per_lb", "city-mpg"]].describe().T.round(2))

## 4. 規格怎麼拉開標價（seaborn）

先收回舊 V2 那幾張：品牌箱型、相關熱圖、排量／馬力／油耗散點、保險評級 × 品牌。小表不用抽樣。

In [ ]:
order = spec.groupby("make")["price"].median().sort_values().index
fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(data=spec, x="price", y="make", order=order, showfliers=False, ax=ax)
ax.set_title("Sticker price by make (Automobile, n=priced)")
plt.tight_layout()
plt.show()

In [ ]:
feats = [
    "price", "log_price", "engine-size", "disp_from_geom", "horsepower",
    "hp_per_lb", "curb-weight", "city-mpg", "highway-mpg", "symboling",
    "make_freq",
]
corr = spec[feats].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Spec features vs sticker price")
plt.tight_layout()
plt.show()
print(corr["price"].drop("price").abs().sort_values(ascending=False).round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
pairs = [
    ("engine-size", "Engine size (c.i.)"),
    ("horsepower", "Horsepower"),
    ("city-mpg", "City mpg"),
]
for ax, (x, title) in zip(axes, pairs):
    sns.regplot(
        data=spec, x=x, y="price", order=2, scatter_kws={"s": 18, "alpha": 0.45},
        line_kws={"color": "C3"}, ax=ax,
    )
    ax.set_title(title)
plt.suptitle("Price vs spec (quadratic overlay)", y=1.02)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.scatterplot(
    data=spec.dropna(subset=["disp_from_geom"]),
    x="engine-size", y="disp_from_geom", hue="num_cylinders", palette="deep", s=28, ax=ax,
)
lims = [
    spec["engine-size"].min(),
    max(spec["engine-size"].max(), spec["disp_from_geom"].max()),
]
ax.plot(lims, lims, ls="--", c="0.5", label="y = x")
ax.set_title("Catalog engine-size vs bore-stroke geometry")
ax.legend(title="cylinders")
plt.tight_layout()
plt.show()

In [ ]:
heat = spec.pivot_table(index="make", columns="symboling", values="price", aggfunc="mean")
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
sns.heatmap(heat, cmap="YlOrRd", ax=axes[0])
axes[0].set_title("Mean sticker by make x symboling")

sns.scatterplot(
    data=spec, x="horsepower", y="city-mpg",
    hue="engine_bin", style="hp_bin", s=36, ax=axes[1],
)
axes[1].set_title("HP vs city mpg (bins from cut)")
plt.tight_layout()
plt.show()

## 5. 互動（plotly）

205 點可以一次丟進去。舊本用 Altair 框選連動長條；這次用 hover + 顏色／分面。沒裝 plotly 就退回 seaborn。

In [ ]:
if HAS_PLOTLY:
    fig = px.scatter(
        spec,
        x="engine-size",
        y="price",
        color="horsepower",
        symbol="body-style",
        hover_data=["make", "city-mpg", "curb-weight", "symboling"],
        title="All priced rows: engine vs sticker (colour = HP, symbol = body)",
    )
    fig.update_traces(marker={"size": 9, "opacity": 0.75})
    fig.show()

    fig = px.scatter(
        spec,
        x="horsepower",
        y="city-mpg",
        color="make",
        facet_col="body-style",
        facet_col_wrap=3,
        hover_data=["price", "engine-size"],
        title="HP vs city mpg, faceted by body-style",
    )
    fig.update_traces(marker={"size": 8})
    fig.show()
else:
    sns.scatterplot(data=spec, x="engine-size", y="price", hue="horsepower", style="body-style", s=28)
    plt.title("plotly missing — seaborn fallback")
    plt.show()

## 6. 細部：把英國上架檔做成品牌列再接進來

Automobile 沒有 `model`／`year`／`mileage`，對不到同一台車。

能做的：`make` 對上英國 `brand`（Audi、BMW、Mercedes、Toyota、VW），規格表中位馬力／車重，接到英國中位上架價。Ford／Hyundai／Skoda／Vauxhall 對不上。有標價的重疊列約 66 筆，對幾萬列市場，年代也不同——當補充，不當同一批車的特徵。

英國檔的清理規則跟 `car_market_eda.ipynb` 同一套；這本不重做市場五題。

In [ ]:
SNAPSHOT_YEAR = 2020
DATA_DIR = Path("car_data")
BRAND_FILES = {
    "audi": "Audi", "bmw": "BMW", "ford": "Ford", "hyundi": "Hyundai",
    "merc": "Mercedes", "skoda": "Skoda", "toyota": "Toyota",
    "vauxhall": "Vauxhall", "vw": "VW",
}
MAKE_TO_BRAND = {
    "audi": "Audi",
    "bmw": "BMW",
    "mercedes-benz": "Mercedes",
    "toyota": "Toyota",
    "volkswagen": "VW",
}

parts = []
for fname, brand in BRAND_FILES.items():
    d = pd.read_csv(DATA_DIR / f"{fname}.csv")
    d = d.rename(columns={"tax(£)": "tax"})
    d["brand"] = brand
    parts.append(d)
uk = pd.concat(parts, ignore_index=True)
uk = uk.drop_duplicates()
uk = uk[(uk["year"] <= SNAPSHOT_YEAR) & (uk["engineSize"] > 0)].copy()
uk["car_age"] = SNAPSHOT_YEAR - uk["year"]
print("UK listings after clean", len(uk))

spec["brand"] = spec["make"].map(MAKE_TO_BRAND)
spec_brand = (
    spec.dropna(subset=["brand"])
    .groupby("brand")
    .agg(
        n_catalog=("brand", "size"),
        median_hp=("horsepower", "median"),
        median_weight=("curb-weight", "median"),
        median_catalog_price=("price", "median"),
        median_engine_cid=("engine-size", "median"),
    )
)
uk_brand = uk.groupby("brand").agg(
    n_listings=("price", "size"),
    median_list_price=("price", "median"),
    median_engine_l=("engineSize", "median"),
    median_age=("car_age", "median"),
)
aligned = uk_brand.join(spec_brand, how="left")
print("UK brands with no catalog row:", aligned[aligned["n_catalog"].isna()].index.tolist())
display(aligned.round(1))

In [ ]:
overlap = aligned.dropna(subset=["median_hp"]).reset_index()
print("overlap", overlap["brand"].tolist(), "catalog rows", int(overlap["n_catalog"].sum()))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
sns.scatterplot(data=overlap, x="median_hp", y="median_list_price", s=130, ax=axes[0])
for _, r in overlap.iterrows():
    axes[0].text(r["median_hp"] + 1.5, r["median_list_price"], r["brand"], fontsize=9)
axes[0].set_title("Catalog median HP vs UK median list price")
axes[0].set_xlabel("Automobile median horsepower")
axes[0].set_ylabel("UK median list price (GBP)")

sns.barplot(data=overlap, x="brand", y="n_catalog", ax=axes[1], color="steelblue")
axes[1].set_title("Catalog side is thin")
plt.tight_layout()
plt.show()

if HAS_PLOTLY:
    fig = px.scatter(
        overlap,
        x="median_hp",
        y="median_list_price",
        size="n_listings",
        text="brand",
        hover_data=["n_catalog", "median_weight", "median_engine_l", "median_catalog_price"],
        title="Weak align: catalog HP vs UK list price (size = listings)",
    )
    fig.update_traces(textposition="top center")
    fig.show()

## 7. 這輪看到什麼

- 規格表上，拉開目錄價的是排量、馬力、車重；`city-mpg` 往往跟價格反向走，因為省油的車比較小。`disp_from_geom` 跟 `engine-size` 應該很靠近，差比較大的列值得回去查 `bore`／`stroke`／缸數。
- `symboling` 是保險風險評級，不是市場熱度。跟價格的關係可以看，但不能當成英國上架的特徵。
- `make_freq` 沒看價格；`normalized-losses` 缺太多，這輪不當主特徵。
- 弱對齊只有五個品牌。馬力中位較高的品牌，英國中位上架價也傾向較高——這是品牌層共變，不是同一台車對上了。

市場五題、折舊、燃料佔比、**同一品牌裡的車型**，看 `car_market_eda.ipynb`。